In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from brain_image.model.eeg_encoder import atms

model = atms.AtmsEEGEncoder()

In [3]:
import torch

model_idx = 1

if model_idx == 0:
    cp = torch.load("outputs/eeg_training/251008_134144/checkpoints/epoch_29-val_loss_5.4749.pt")
    state_dict = cp["model"]
    pass

elif model_idx == 1:
    cp = torch.load("tmp/40.pth")
    cp = {k: v for k, v in cp.items() if not (k.startswith("subject_wise") or "subject_embedding" in k)}
    state_dict = cp
    pass

elif model_idx == 2:
    cp = torch.load("outputs/eeg_training/251008_150716/checkpoints/epoch_24-val_loss_2.0365.pt")
    state_dict = cp["model"]
    pass

model.load_state_dict(state_dict)

model.eval()
model.requires_grad_(False)

AtmsEEGEncoder(
  (encoder): iTransformer(
    (enc_embedding): DataEmbedding(
      (value_embedding): Linear(in_features=250, out_features=250, bias=True)
      (position_embedding): PositionalEmbedding()
      (temporal_embedding): TimeFeatureEmbedding(
        (embed): Linear(in_features=4, out_features=250, bias=False)
      )
      (dropout): Dropout(p=0.25, inplace=False)
    )
    (encoder): Encoder(
      (attn_layers): ModuleList(
        (0): EncoderLayer(
          (attention): AttentionLayer(
            (inner_attention): FullAttention(
              (dropout): Dropout(p=0.25, inplace=False)
            )
            (query_projection): Linear(in_features=250, out_features=248, bias=True)
            (key_projection): Linear(in_features=250, out_features=248, bias=True)
            (value_projection): Linear(in_features=250, out_features=248, bias=True)
            (out_projection): Linear(in_features=248, out_features=250, bias=True)
          )
          (conv1): Conv1d

In [4]:
from pathlib import Path
from typing import Literal

from brain_image.data import TensorCache


class EEGDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        split: Literal["train", "test"],
        data_path: Path = Path("data/things-eeg2"),
        sub: int = 8,
        img_encoder: str = "clip_vith14",
    ):
        self.split = split
        self.data_path = data_path
        self.sub = sub
        self.img_encoder = img_encoder

        self._tensorcache = TensorCache(memory_cache_size=128000)
        self.data = torch.load(data_path / "prepared" / f"sub-{sub:02}" / f"{split}.pt")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx: int):
        sample = self.data[idx]
        img_path = sample["img_path"]

        return {
            "img_path": img_path,
            "eeg_data": sample["eeg"],
            "img_embedding": self._tensorcache.get(
                img_path, self.img_encoder, f"{self.split}.pt"
            ),
            "idx": sample["idx"],
            "sub": sample["sub"],
        }

ds = EEGDataset("test")
eeg = torch.stack(list(ds[i]["eeg_data"] for i in range(64)))
imgf = torch.stack(list(ds[i]["img_embedding"] for i in range(64)))

In [5]:
from brain_image.model.img_encoder import load_image_encoder


img_enc = load_image_encoder("clip_vith14")

In [6]:
from brain_image.data import batch_load_images


imgs = batch_load_images(Path(ds[i]["img_path"]) for i in range(64))


In [7]:
their_imgfs= torch.load("tmp/ViT-H-14_features_test.pt")["img_features"]
timgf = their_imgfs[:64]
their_imgfs

tensor([[ 0.0124, -0.0171,  0.0036,  ...,  0.0219, -0.0223,  0.0142],
        [ 0.0240,  0.0055,  0.0353,  ..., -0.0066, -0.0064, -0.0013],
        [-0.0428, -0.0135,  0.0133,  ...,  0.0238, -0.0275,  0.0193],
        ...,
        [ 0.0048, -0.0178,  0.0163,  ...,  0.0010, -0.0104,  0.0328],
        [-0.0239,  0.0144, -0.0109,  ..., -0.0110, -0.0177,  0.0282],
        [ 0.0256,  0.0178,  0.0134,  ...,  0.0267, -0.0028,  0.0616]])

In [14]:
import math
eegf = model(eeg) 
neegf = torch.nn.functional.normalize(eegf)
nimgf = torch.nn.functional.normalize(imgf, p=2, dim=-1)
def investigate_t(t):
    print(t.mean(dim=-1).mean(), t.std(dim=-1).mean(), t.norm(dim=-1).mean())
investigate_t(eegf)         # unnormalized eeg embedfdinsg
investigate_t(neegf)        # Normaalizewd eeg embeddings
investigate_t(imgf)         # Image embeddings (clip-H14), not normalized
investigate_t(nimgf)        # My image embeddings, normalized
investigate_t(timgf * math.sqrt(1024))        # Their image embeddings, normalized


tensor(4.8900e-05) tensor(0.1466) tensor(4.6891)
tensor(8.3043e-06) tensor(0.0313) tensor(1.)
tensor(-0.0037) tensor(0.6951) tensor(22.2407)
tensor(-0.0002) tensor(0.0313) tensor(1.)
tensor(-0.0020) tensor(1.0001) tensor(32.)


In [9]:
torch.allclose(timgf, nimgf, rtol=0.001)

False